# Task 3 a–d: segmentasi pelanggan
Muhammad Habibie Rabbani — 101032300005 — Kelompok 11

Klik Run All dengan kernel proyek. Notebook mandiri ini mencakup audit/preprocessing,
pemilihan fitur, tiga metode, dan pemilihan k berdasarkan silhouette serta Davies–Bouldin.
Bagian e (stabilitas resampling), f (profil/aksi), dan g (anomali) belum dikerjakan.
Tidak menghasilkan cluster.csv final. Dataset mentah tetap di luar repository.


In [ ]:
from pathlib import Path
import json, hashlib, time, platform
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, cut_tree
from threadpoolctl import threadpool_limits

SEED = 42
N_FIT = 6000
N_SCORE = 1500
K_GRID = list(range(2, 13))
start = Path.cwd().resolve()
REPO = next((p for p in [start, *start.parents] if (p/'kode/Task3_Clustering.ipynb').is_file()), None)
if REPO is None: raise FileNotFoundError('Buka notebook dari folder repository atau kode.')
DATA_DIR = Path(globals().get('DATA_DIR', REPO.parent))
OUT = REPO/'kode/hasil/task3_ad'
OUT.mkdir(parents=True, exist_ok=True)
path = DATA_DIR/'clustering.csv.gz'
data = pd.read_csv(path)
assert data.id_pelanggan.notna().all() and data.id_pelanggan.is_unique
rng = np.random.default_rng(SEED)
idx = np.sort(rng.choice(len(data), min(N_FIT,len(data)), replace=False))
score_idx = np.sort(rng.choice(len(idx), min(N_SCORE,len(idx)), replace=False))
print('Dataset:',data.shape,'; sampel perbandingan:',len(idx))


## a. Audit dan preprocessing
Audit dilakukan pada semua pelanggan. Nilai ekstrem tidak otomatis dianggap salah atau
anomali. Untuk geometri clustering, numerik dibatasi pada persentil 0,5–99,5 lalu
log1p jika nonnegatif dan skewness >1; imputasi median dan standardisasi.
Nilai sumber tetap utuh, jumlah nilai yang dibatasi dicatat.

Dua blok komposisi ditutup kembali agar jumlahnya satu, kemudian diubah dengan akar
kuadrat (geometri Hellinger). Nol sah sehingga tidak perlu pseudocount. Dua kolom
kategori memakai one-hot. Setiap blok dinormalisasi dengan akar rata-rata energi
kuadrat agar jumlah kolom tidak otomatis menentukan pengaruhnya.


In [ ]:
cat_share = [c for c in data if c.startswith('porsi_') and not c.startswith('porsi_jam_')]
time_share = [c for c in data if c.startswith('porsi_jam_')]
crm = [c for c in data if c.startswith('n') and c[1:].isdigit()]
profile_num = ['lama_member_bulan','umur']
profile_cat = ['kota_tier','perangkat']
behavior = [c for c in data.select_dtypes('number') if c not in cat_share+time_share+crm+profile_num+profile_cat]
audit = data.describe(include='all').T
numeric = data.select_dtypes('number')
audit['missing'] = data.isna().sum()
audit['skew'] = numeric.skew()
audit.to_csv(OUT/'audit_kolom.csv', index_label='fitur')
compositions = []
for name,cols in [('kategori',cat_share),('waktu',time_share)]:
    x=data[cols]
    compositions.append({'blok':name,'kolom':len(cols),'negatif':int((x<0).sum().sum()),
       'baris_missing':int(x.isna().any(axis=1).sum()),'jumlah_nol':int((x.sum(axis=1)==0).sum()),
       'deviasi_jumlah_max':float((x.sum(axis=1)-1).abs().max()),
       'baris_deviasi_diatas_001':int(((x.sum(axis=1)-1).abs()>.001).sum())})
pd.DataFrame(compositions).to_csv(OUT/'audit_komposisi.csv',index=False)
summary = {'baris':len(data),'kolom':len(data.columns),'id_duplikat':int(data.id_pelanggan.duplicated().sum()),
 'baris_duplikat_tanpa_id':int(data.drop(columns='id_pelanggan').duplicated().sum()),
 'rasio_konversi_diatas_1':int((data.rasio_konversi>1).sum()),
 'recency_diatas_90':int((data.recency_hari>90).sum()),
 'crm_di_luar_kamus':[c for c in crm if int(c[1:])>10]}
(OUT/'audit_ringkasan.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
display(pd.DataFrame(compositions));print(summary)

# Data tak berlabel: parameter preprocessing dipelajari dari populasi yang disegmentasi.
# Skor berikut deskriptif, bukan estimasi generalisasi out-of-sample.
def numeric_block(frame, indicator=True):
    x=frame.astype(float).replace([np.inf,-np.inf],np.nan)
    rows=[]
    for c in x:
        lo,hi=x[c].quantile([.005,.995]);skew=x[c].skew()
        use_log=bool(x[c].min()>=0 and skew>1)
        rows.append({'fitur':c,'batas_bawah':lo,'batas_atas':hi,'log1p':use_log,
                     'jumlah_dibatasi':int(((x[c]<lo)|(x[c]>hi)).sum()),'missing':int(x[c].isna().sum())})
        x[c]=x[c].clip(lo,hi)
        if use_log:x[c]=np.log1p(x[c])
    imp=SimpleImputer(strategy='median',add_indicator=indicator,keep_empty_features=True)
    z=StandardScaler().fit_transform(imp.fit_transform(x))
    return z,rows,x

def composition_block(frame):
    x=frame.astype(float).copy()
    if (x<0).any().any():raise ValueError('Porsi negatif perlu ditinjau sebelum transformasi.')
    x=x.fillna(x.median()).to_numpy()
    totals=x.sum(axis=1,keepdims=True)
    x=np.divide(x,totals,out=np.full_like(x,1/x.shape[1]),where=totals>0)
    return np.sqrt(x)

def weight_block(z,weight=1.):
    z=z-z.mean(axis=0)
    scale=np.sqrt(np.mean(np.sum(z*z,axis=1)))
    return z*(np.sqrt(weight)/max(scale,1e-12))

z_behavior,rows,transformed=numeric_block(data[behavior])
z_profile,rows_p,_=numeric_block(data[profile_num])
z_crm,rows_c,_=numeric_block(data[crm])
encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
z_cat=encoder.fit_transform(data[profile_cat].astype('string').fillna('__missing__'))
z_share=composition_block(data[cat_share]);z_time=composition_block(data[time_share])
pd.DataFrame(rows+rows_p+rows_c).to_csv(OUT/'parameter_preprocessing.csv',index=False)
assert np.allclose(np.sum(z_share*z_share,axis=1),1)
assert np.isfinite(z_behavior).all()


## b. Pemilihan dan bobot fitur
ID dikeluarkan. Fitur anonim CRM dikeluarkan dari representasi utama karena maknanya
belum diketahui dan n11–n15 tidak tercantum dalam kamus; ini bukan bukti bahwa fitur
tersebut tidak berguna. Dampaknya diuji dengan ablation.

Dari fitur perilaku, buang satu fitur pada pasangan korelasi absolut >0,95 setelah
transformasi, dengan urutan kolom sebagai aturan deterministik. Semua pasangan
berkorelasi dan alasan keputusan disimpan. Komposisi tidak diseleksi per kolom.
Bobot utama: perilaku 1, komposisi kategori 1, komposisi waktu 1, profil numerik 0,25,
profil kategori 0,25. Bobot profil lebih kecil agar perilaku dominan; ini pilihan
pemodelan yang diuji terhadap bobot profil 1 dan penambahan CRM 1.


In [ ]:
corr=transformed.corr().abs();kept=[];dropped={};pairs=[]
for c in behavior:
    conflicts=[k for k in kept if corr.loc[c,k]>.95]
    if conflicts:dropped[c]=conflicts[0]
    else:kept.append(c)
for i,c in enumerate(behavior):
    for other in behavior[i+1:]:
        if corr.loc[c,other]>.8:pairs.append({'fitur_1':c,'fitur_2':other,'abs_korelasi':corr.loc[c,other]})
pd.DataFrame(pairs,columns=['fitur_1','fitur_2','abs_korelasi']).to_csv(OUT/'korelasi_perilaku.csv',index=False)
z_kept,_,_=numeric_block(data[kept])
blocks=[weight_block(z_kept),weight_block(z_share),weight_block(z_time)]
X=np.column_stack(blocks+[weight_block(z_profile,.25),weight_block(z_cat,.25)]).astype('float32')
feature_rows=[]
for c in data:
    if c=='id_pelanggan':decision='dikeluarkan: identitas'
    elif c in crm:decision='dikeluarkan utama: anonim; diuji ablation'
    elif c in dropped:decision='dikeluarkan: redundan dengan '+dropped[c]
    else:decision='dipakai'
    feature_rows.append({'fitur':c,'keputusan':decision})
pd.DataFrame(feature_rows).to_csv(OUT/'keputusan_fitur.csv',index=False)
assert np.isfinite(X).all() and len(X)==len(data)
print('Fitur redundan:',dropped,'; dimensi hasil:',X.shape[1])


## c–d. Perbandingan metode dan jumlah cluster
Bandingkan MiniBatch K-Means, Gaussian Mixture diagonal, dan hierarchical Ward.
Ketiganya dilatih pada 6.000 pelanggan yang sama dengan seed 42; k=2..12.
Ward menggunakan jarak Euclidean pada representasi yang sama. Pembatasan sampel
membuat Ward layak dijalankan, tetapi pelanggan langka mungkin tidak terwakili.

Silhouette dihitung pada 1.500 pelanggan tetap; Davies–Bouldin dan Calinski–Harabasz
pada seluruh sampel fit. BIC hanya dibandingkan antar-k untuk GMM dengan data dan
representasi yang sama. Pemilihan kandidat memakai rata-rata peringkat silhouette
(lebih besar lebih baik) dan DB (lebih kecil lebih baik); tie dipecahkan dengan k
lebih kecil. Ini heuristik internal, bukan bukti jumlah segmen sebenarnya.


In [ ]:
sample=X[idx];score=sample[score_idx]
results=[];partitions={}
with threadpool_limits(limits=4):
    tree=linkage(sample,method='ward',metric='euclidean')
    for k in K_GRID:
        for method in ['MiniBatchKMeans','GaussianMixture_diag','Ward']:
            tick=time.monotonic();bic=np.nan;converged=True
            if method=='MiniBatchKMeans':
                model=MiniBatchKMeans(n_clusters=k,n_init=10,batch_size=1024,random_state=SEED)
                labels=model.fit_predict(sample)
            elif method=='GaussianMixture_diag':
                model=GaussianMixture(n_components=k,covariance_type='diag',n_init=3,max_iter=300,reg_covar=1e-5,random_state=SEED)
                labels=model.fit_predict(sample);bic=model.bic(sample);converged=bool(model.converged_)
            else:labels=cut_tree(tree,n_clusters=k).ravel()
            counts=np.bincount(labels,minlength=k)
            scored=labels[score_idx]
            sil=silhouette_score(score,scored) if 1<len(np.unique(scored))<len(scored) else np.nan
            results.append({'metode':method,'k':k,'silhouette':sil,'davies_bouldin':davies_bouldin_score(sample,labels),
                'calinski_harabasz':calinski_harabasz_score(sample,labels),'bic':bic,'konvergen':converged,
                'cluster_terkecil':int(counts.min()),'cluster_terbesar':int(counts.max()),'detik':time.monotonic()-tick})
            partitions[(method,k)]=labels
            print(method,k,'silhouette',round(sil,4),flush=True)
metrics=pd.DataFrame(results)
metrics['layak']=metrics.konvergen & metrics.silhouette.notna() & (metrics.cluster_terkecil>0)
metrics['rank_silhouette']=metrics.loc[metrics.layak,'silhouette'].rank(ascending=False)
metrics['rank_db']=metrics.loc[metrics.layak,'davies_bouldin'].rank()
metrics['rank_rata']=(metrics.rank_silhouette+metrics.rank_db)/2
metrics.to_csv(OUT/'perbandingan_metode_k.csv',index=False)
chosen=metrics[metrics.layak].sort_values(['rank_rata','k','metode']).iloc[0]
print('Kandidat a-d:',chosen[['metode','k','silhouette','davies_bouldin']].to_dict())
fig,axes=plt.subplots(1,2,figsize=(11,4))
for name,g in metrics.groupby('metode'):
    axes[0].plot(g.k,g.silhouette,'o-',label=name)
    axes[1].plot(g.k,g.davies_bouldin,'o-',label=name)
for ax,title in zip(axes,['Silhouette (lebih tinggi lebih baik)','Davies–Bouldin (lebih rendah lebih baik)']):
    ax.set(xlabel='Jumlah cluster k',title=title);ax.grid(alpha=.2);ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(OUT/'pemilihan_k.png',dpi=150);
if 'agg' != plt.get_backend().lower(): plt.show()
plt.close(fig)


## Bukti sensitivitas fitur dan keputusan sementara
Ablation memakai MiniBatch K-Means dan GMM diagonal pada k kandidat yang sama. Setiap partisi dinilai
pada representasi utama yang sama agar skor lintas pilihan fitur memiliki acuan
jarak yang tetap. ARI di sini mengukur perubahan partisi akibat fitur; ini belum
uji stabilitas bootstrap/subsampling bagian e. Tidak mengganti pilihan setelah
melihat ablation; ketidakkonsistenan dicatat sebagai batasan untuk diskusi bersama.


In [ ]:
z_no_indicator,_,_=numeric_block(data[kept], indicator=False)
variants={'utama':X,
 'tanpa_indikator_missing':np.column_stack([weight_block(z_no_indicator),*blocks[1:],weight_block(z_profile,.25),weight_block(z_cat,.25)]),
 'tanpa_seleksi_korelasi':np.column_stack([weight_block(z_behavior),*blocks[1:],weight_block(z_profile,.25),weight_block(z_cat,.25)]),
 'profil_bobot_1':np.column_stack(blocks+[weight_block(z_profile),weight_block(z_cat)]),
 'tambah_crm':np.column_stack([X,weight_block(z_crm)])}
k=int(chosen.k);ab=[]
with threadpool_limits(limits=4):
    for method in ['MiniBatchKMeans','GaussianMixture_diag']:
        reference=partitions[(method,k)]
        for name,z in variants.items():
            if name=='utama': labels=reference
            elif method=='MiniBatchKMeans': labels=MiniBatchKMeans(n_clusters=k,n_init=10,batch_size=1024,random_state=SEED).fit_predict(z[idx])
            else: labels=GaussianMixture(n_components=k,covariance_type='diag',n_init=3,max_iter=300,reg_covar=1e-5,random_state=SEED).fit_predict(z[idx])
            ab.append({'metode':method,'representasi':name,'k':k,'silhouette_acuan_utama':silhouette_score(score,labels[score_idx]),
                'db_acuan_utama':davies_bouldin_score(sample,labels),'ari_vs_utama':adjusted_rand_score(reference,labels)})
ablation=pd.DataFrame(ab);ablation.to_csv(OUT/'ablation_fitur.csv',index=False);display(ablation)
labels=partitions[(chosen.metode,k)]
pd.DataFrame({'id_pelanggan':data.id_pelanggan.iloc[idx].to_numpy(),'cluster_kandidat':labels}).to_csv(OUT/'partisi_sampel_eksploratif.csv',index=False)
diagnostic=pd.DataFrame({'cluster_kandidat':labels,'basket_missing':data.rata2_keranjang.iloc[idx].isna().to_numpy(),
                        'frekuensi_nol':(data.frekuensi_90h.iloc[idx]==0).to_numpy()}).groupby('cluster_kandidat').agg(
                        jumlah=('basket_missing','size'),porsi_basket_missing=('basket_missing','mean'),porsi_frekuensi_nol=('frekuensi_nol','mean'))
diagnostic.to_csv(OUT/'diagnostik_missing_cluster.csv')
gmm_bic=metrics[metrics.metode=='GaussianMixture_diag'].sort_values('bic').iloc[0]
meta={'scope':'Task 3 a-d; kandidat, belum output final e-g','seed':SEED,'n_fit':len(idx),'n_score':len(score_idx),
 'k_grid':K_GRID,'metode_kandidat':str(chosen.metode),'k_kandidat':k,
 'aturan':'rata-rata rank silhouette dan Davies-Bouldin pada sampel tetap; tie k kecil',
 'batas_grid':bool(k in [min(K_GRID),max(K_GRID)]),
 'sha256_input':hashlib.sha256(path.read_bytes()).hexdigest(),'python':platform.python_version(),
 'versi':{p:version(p) for p in ['numpy','pandas','scipy','scikit-learn','matplotlib']},
 'batasan':['Preprocessing pada populasi penuh tanpa label; evaluasi deskriptif, bukan holdout.',
 'Fit dan evaluasi metode memakai sampel yang sama; skor internal optimistis.',
 'Belum stabilitas resampling, profil tindakan, atau penandaan anomali.',
 'CRM anonim dikecualikan secara konservatif; bukan bukti semua fitur CRM tidak relevan.',
 'Partisi sampel tidak boleh dikumpulkan sebagai cluster.csv final.']}
(OUT/'keputusan_ad.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
def table(frame):
    return '| '+' | '.join(frame.columns)+' |\n| '+' | '.join(['---']*len(frame.columns))+' |\n'+'\n'.join('| '+' | '.join(str(v) for v in row)+' |' for row in frame.itertuples(index=False,name=None))
report=['# Task 3 a-d — draf analisis',
 'Audit seluruh 120.000 pelanggan; model dibandingkan pada sampel tetap. Tidak ada label referensi tersembunyi yang digunakan.',
 '## a. Audit dan preprocessing',json.dumps(summary,ensure_ascii=False),
 'Imputasi median, winsorisasi 0,5–99,5 persen, log1p untuk numerik nonnegatif miring, standardisasi; komposisi memakai closure + akar kuadrat; kategori one-hot. Semua pelanggan dipertahankan. Winsorisasi tidak menetapkan label anomali.',
 'Nilai recency >90 bukan otomatis salah karena pelanggan dapat lama tidak bertransaksi. Rasio konversi >1 dicatat sebagai anomali definisi/kualitas, tidak langsung dibuang. Detail tersimpan pada audit_kolom.csv dan parameter_preprocessing.csv.',
 '## b. Fitur', 'ID dikeluarkan; n01–n15 hanya untuk ablation. n11–n15 tidak tercantum dalam kamus. Profil diberi bobot 0,25 per blok; perilaku dan kedua komposisi bobot 1. Korelasi >0,95 ditangani pada blok perilaku saja. Fitur dibuang: '+str(dropped), table(ablation.round(5)),
 'Skor ablation memakai jarak acuan utama yang sama. ARI di sini sensitivitas fitur, bukan stabilitas resampling.',
 '## c-d. Metode dan jumlah cluster',table(metrics.drop(columns=['detik']).round(5)),
 '![Pemilihan k](pemilihan_k.png)',
 f'Kandidat: {chosen.metode}, k={k}. Pemilihan menggunakan rata-rata peringkat silhouette dan DB; CH dan BIC GMM menjadi diagnostik tambahan. BIC tidak dibandingkan antaralgoritme.',
 'Ward memakai satu dendrogram untuk seluruh k; waktu per-k Ward di CSV hanya pemotongan pohon dan scoring, sehingga tidak boleh dibandingkan sebagai waktu training total antaralgoritme.',
 'Kandidat berada di batas grid: '+str(meta['batas_grid'])+'. Jika benar, rentang k perlu ditinjau sebelum kesimpulan final.',
 '## Pemeriksaan artefak missing dan ketidaksepakatan k',table(diagnostic.reset_index()),
 'Dua cluster kandidat memisahkan ada/tidaknya rata2_keranjang. Ini dapat mencerminkan tidak adanya transaksi, tetapi juga dipengaruhi indikator missing. Uji tanpa indikator tersedia pada ablation_fitur.csv. Jangan menyebut dua cluster ini jumlah segmen pemasaran sebenarnya sebelum bagian e-g.',
 f'BIC GMM paling kecil pada k={int(gmm_bic.k)}, berbeda dari pilihan gabungan silhouette/DB bila nilainya tidak sama. Kriteria internal dapat bertentangan. Silhouette kandidat hanya {chosen.silhouette:.4f}; struktur kelompok belum kuat.',
 'Ablation dinilai pada ruang utama yang memang dirancang dengan pilihan fitur tersebut, sehingga tidak membuktikan fitur alternatif buruk atau tidak berguna. Bobot dan pemilihan fitur masih perlu ditinjau bersama.',
 '## Keterbatasan dan kelanjutan',*meta['batasan'],
 'Tahap berikutnya e–g dilakukan bersama: stabilitas bootstrap/subsampling dengan ARI, profil/aksi pemasaran, dan aturan anomali -1. cluster.csv sementara dibuat dari kandidat a-d untuk integrasi, lalu harus diganti sebelum pengumpulan final.',
 '## Reproduksi dan AI','Jalankan seluruh cell kode/Task3_Clustering.ipynb. Seed, versi, dan fingerprint input ada di keputusan_ad.json. Codex membantu menulis kode, tes pemeriksaan, eksperimen dan draf. Anggota wajib memeriksa hasil dan memahami keputusan.']
(OUT/'laporan_task3_ad.md').write_text('\n\n'.join(report)+'\n',encoding='utf-8')
print('Selesai a-d. Kandidat belum final. Hasil:',OUT)


## cluster.csv sementara untuk integrasi kelompok
Kandidat GMM diagonal k=2 dari sampel a–d dipakai memprediksi semua pelanggan.
File ini **sementara**, belum menjalani uji stabilitas, profil segmen, atau aturan anomali `-1`.
Ganti file ini setelah bagian e–g sebelum pengumpulan final.

In [ ]:
if chosen.metode != 'GaussianMixture_diag':
    raise ValueError('Output sementara hanya tersedia untuk kandidat GMM diagonal.')
with threadpool_limits(limits=4):
    model_sementara = GaussianMixture(n_components=k, covariance_type='diag', n_init=3, max_iter=300, reg_covar=1e-5, random_state=SEED)
    model_sementara.fit(sample)
    label_sampel = model_sementara.predict(sample)
    label_semua = model_sementara.predict(X)
assert np.array_equal(label_sampel, partitions[(chosen.metode, k)])
cluster_sementara = pd.DataFrame({'id_pelanggan': data.id_pelanggan, 'cluster': label_semua.astype(int)})
assert len(cluster_sementara) == len(data) and cluster_sementara.id_pelanggan.is_unique
assert cluster_sementara.cluster.between(0, k - 1).all()
cluster_sementara.to_csv(REPO/'cluster.csv', index=False)
(OUT/'cluster_sementara_meta.json').write_text(json.dumps({'status': 'SEMENTARA; bukan jawaban final', 'metode': str(chosen.metode), 'k': k, 'n_fit': len(idx), 'n_prediksi': len(data), 'anomali_minus_satu': False, 'seed': SEED, 'sha256_input': meta['sha256_input']}, indent=2), encoding='utf-8')
print('cluster.csv SEMENTARA:', len(cluster_sementara), 'pelanggan; ukuran cluster:', cluster_sementara.cluster.value_counts().sort_index().to_dict())
